# Pipeline de Engenharia de Dados — Cidades Inteligentes
## Demonstração com Dados Reais: Bronze → Silver → Gold

> **Aluno:** Gabriel Felice | **Disciplina:** Engenharia de Dados | **Data:** Junho de 2026

Este notebook complementa o documento `README.md` com exemplos práticos de:
- Dados no formato real de cada fonte (Camada Bronze)
- Transformações de limpeza e padronização (Bronze → Silver)
- Feature engineering e preparação para IA (Silver → Gold)
- Visualização exploratória dos dados em cada camada

---
## 0. Setup e Dependências

In [ ]:
# Instalar dependências (execute uma vez)
# !pip install pandas numpy matplotlib seaborn faker requests python-dateutil pyarrow

import pandas as pd
import numpy as np
import json
import re
import hashlib
import warnings
from datetime import datetime, timedelta, timezone
from dateutil import parser as dateparser

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

# Seed para reprodutibilidade
np.random.seed(42)

print("Setup concluído.")
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")

---
## 1. CAMADA BRONZE — Simulação de Dados Brutos

Os dados abaixo refletem fielmente os formatos e problemas reais descritos no documento de arquitetura.
Os padrões de dados são baseados em:
- **Sensores de tráfego:** Padrão CET-SP (Centro de Engenharia de Tráfego)
- **Qualidade do ar:** Padrão CETESB (Companhia Ambiental do Estado de São Paulo)
- **GPS de ônibus:** API SPTrans
- **Ouvidoria:** Modelo do sistema 156 SP
- **Meteorologia:** API INMET (dados públicos reais — inmet.gov.br)

In [ ]:
# ============================================================
# BRONZE — Fonte 1: Sensores IoT de Tráfego
# Simula payloads MQTT de 4.200 sensores ao longo de 24h
# ============================================================

VIAS = [
    ("Av. Paulista",       "Norte-Sul",    -23.5613, -46.6547),
    ("Av. Paulista",       "Sul-Norte",    -23.5613, -46.6547),
    ("Marginal Tietê",     "Leste-Oeste",  -23.5125, -46.6395),
    ("Marginal Pinheiros", "Norte-Sul",    -23.5745, -46.6992),
    ("Av. 23 de Maio",     "Norte-Sul",    -23.5847, -46.6404),
    ("Radial Leste",       "Leste-Oeste",  -23.5441, -46.5957),
    ("Av. Brasil",         "Leste-Oeste",  -23.5341, -46.6290),
    ("Corredor ABD",       "Norte-Sul",    -23.7030, -46.5552),
]

def simular_sensores_bronze(n_registros=500):
    registros = []
    base_time = datetime(2026, 6, 19, 6, 0, 0, tzinfo=timezone.utc)
    
    for i in range(n_registros):
        via_info = VIAS[i % len(VIAS)]
        via, sentido, lat, lon = via_info
        
        sensor_idx = (i // len(VIAS)) + 1
        sensor_id = f"CET-SP-{(i % 200 + 800):04d}"
        
        # Simula padrão temporal: pico às 8h e 18h
        hora = (6 + (i * 30 // 3600)) % 24
        fator_pico = 1 + 1.5 * (np.exp(-((hora - 8) ** 2) / 4) + np.exp(-((hora - 18) ** 2) / 4))
        
        contagem = int(np.random.poisson(30 * fator_pico))
        velocidade = max(5, np.random.normal(45 / fator_pico, 8))
        ocupacao = min(100, np.random.normal(40 * fator_pico, 10))
        
        # Introduz problemas propositais (como nas fontes reais)
        timestamp = base_time + timedelta(seconds=i * 30)
        
        # Bug 1: ~5% de sensores em manutenção
        if np.random.random() < 0.05:
            contagem = -1
            velocidade = None
        
        # Bug 2: ~3% com velocidade nula (sem veículos)
        elif contagem == 0:
            velocidade = None
        
        # Bug 3: ~2% com timestamp em BRT (horário local) sem aviso
        ts_str = timestamp.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        if np.random.random() < 0.02:
            ts_str = (timestamp - timedelta(hours=3)).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        else:
            ts_str += "Z"
        
        carros = int(contagem * 0.80) if contagem > 0 else 0
        motos  = int(contagem * 0.13) if contagem > 0 else 0
        onibus = max(0, contagem - carros - motos) if contagem > 0 else 0
        
        registros.append({
            "sensor_id":            sensor_id,
            "lat":                  round(lat + np.random.normal(0, 0.001), 6),
            "lon":                  round(lon + np.random.normal(0, 0.001), 6),
            "timestamp_utc":        ts_str,
            "via":                  via,
            "sentido":              sentido,
            "contagem_veiculos":    contagem,
            "velocidade_media_kmh": round(velocidade, 1) if velocidade is not None else None,
            "ocupacao_pct":         round(ocupacao, 1) if contagem != -1 else None,
            "classificacao":        ["carro", "moto", "onibus"],
            "contagem_por_classe":  [carros, motos, onibus],
            "firmware_version":     "2.1.4",
            "bateria_pct":          np.random.randint(60, 100),
            "_ingested_at":         datetime.utcnow().isoformat() + "Z",
            "_source_system":       "iot_traffic_cet",
        })
    
    return pd.DataFrame(registros)

df_traffic_bronze = simular_sensores_bronze(500)

print("=" * 60)
print("BRONZE — Sensores de Tráfego")
print(f"Registros: {len(df_traffic_bronze):,}")
print(f"Problemas introduzidos:")
print(f"  - Sensores em manutenção (contagem=-1): {(df_traffic_bronze.contagem_veiculos == -1).sum()}")
print(f"  - Velocidade nula: {df_traffic_bronze.velocidade_media_kmh.isna().sum()}")
print("=" * 60)
df_traffic_bronze.head(3)

In [ ]:
# ============================================================
# BRONZE — Fonte 2: Qualidade do Ar (padrão CETESB)
# CSV com ponto e vírgula, datas em DD/MM/YYYY, campos vazios
# ============================================================

ESTACOES = [
    ("PIN001", "Pinheiros",       -23.5618, -46.7027),
    ("OSA001", "Osasco",          -23.5329, -46.7920),
    ("IBI001", "Ibirapuera",      -23.5874, -46.6576),
    ("SAO001", "Santo André",     -23.6640, -46.5322),
    ("CAC001", "Caçapava",        -23.1006, -45.7086),
    ("GUA001", "Guarulhos",       -23.4543, -46.5330),
    ("MOO001", "Mogi das Cruzes", -23.5230, -46.1920),
]

def simular_ar_bronze_csv(n_horas=48):
    linhas_csv = ["ESTACAO_ID;DT_MEDICAO;HORA;MP10_ug_m3;MP2.5_ug_m3;O3_ug_m3;NO2_ug_m3;CO_mg_m3;SO2_ug_m3;TEMP_C;UMIDADE_PCT;PRESSAO_hPa;VENTO_DIR_GRAUS;VENTO_VEL_ms"]
    
    base_date = datetime(2026, 6, 18, 0, 0, 0)
    
    for h in range(n_horas):
        dt = base_date + timedelta(hours=h)
        hora_str = dt.strftime("%H:00")
        data_str = dt.strftime("%d/%m/%Y")  # Formato BR — problema proposital
        
        # Padrão diário: poluição maior em horário de pico
        hora = dt.hour
        fator_poluicao = 1 + 0.8 * (np.exp(-((hora - 8) ** 2) / 6) + np.exp(-((hora - 19) ** 2) / 6))
        
        for est_id, est_nome, lat, lon in ESTACOES:
            mp10  = round(np.random.normal(38 * fator_poluicao, 8), 1)
            mp25  = round(np.random.normal(18 * fator_poluicao, 5), 1)
            o3    = round(np.random.normal(80, 20), 1)
            no2   = round(np.random.normal(50 * fator_poluicao, 12), 1)
            co    = round(np.random.normal(0.8 * fator_poluicao, 0.2), 2)
            so2   = round(np.random.normal(4.5, 1.5), 1)
            temp  = round(np.random.normal(19 + 5 * np.sin(hora * np.pi / 12), 1.5), 1)
            umid  = round(np.random.normal(72, 8), 1)
            press = round(np.random.normal(1013, 3), 1)
            vdir  = np.random.randint(0, 360)
            vvel  = round(np.random.exponential(3.5), 1)
            
            # Bug 1: ~8% de campos ausentes (string vazia)
            if np.random.random() < 0.08:
                mp10 = ""
            if np.random.random() < 0.06:
                so2 = ""
            if np.random.random() < 0.04:
                temp = ""
            
            # Bug 2: estação CAC001 usa CO em ppm (não mg/m³)
            if est_id == "CAC001" and co != "":
                co = round(co / 1.165, 3)  # mg/m³ → ppm
            
            # Bug 3: valores negativos esporádicos
            if np.random.random() < 0.01:
                mp25 = -0.3
            
            linha = ";".join([str(x) for x in [
                est_id, data_str, hora_str, mp10, mp25, o3, no2, co, so2,
                temp, umid, press, vdir, vvel
            ]])
            linhas_csv.append(linha)
    
    return "\n".join(linhas_csv)

csv_ar_bronze = simular_ar_bronze_csv(48)

# Carrega como DataFrame para inspeção
from io import StringIO
df_ar_bronze = pd.read_csv(StringIO(csv_ar_bronze), sep=";")

print("=" * 60)
print("BRONZE — Qualidade do Ar (CSV formato CETESB)")
print(f"Registros: {len(df_ar_bronze):,}")
print(f"Campos vazios por coluna:")
print(df_ar_bronze.isin(["", " "]).sum()[df_ar_bronze.isin(["", " "]).sum() > 0])
print(f"\nValores negativos (MP2.5): {(pd.to_numeric(df_ar_bronze['MP2.5_ug_m3'], errors='coerce') < 0).sum()}")
print("=" * 60)
df_ar_bronze.head(3)

In [ ]:
# ============================================================
# BRONZE — Fonte 3: GPS de Ônibus SPTrans
# ============================================================

LINHAS = [
    ("875P-10", "Pinheiros → Centro"),
    ("702U-10", "USP → Butantã"),
    ("6450-10", "Jabaquara → Aeroporto"),
    ("8000-10", "Santo André → SP"),
    ("5100-10", "Lapa → Barra Funda"),
]

LOTACAO_VARIANTS = ["CHEIA", "LOTADA", "CHEIO", "FULL", "MEIA", "VAZIA", "MEIA_LOTACAO"]

def simular_gps_bronze(n_registros=300):
    registros = []
    base_ts = int(datetime(2026, 6, 19, 7, 0, 0).timestamp())
    
    for i in range(n_registros):
        linha_info = LINHAS[i % len(LINHAS)]
        linha_id, descricao = linha_info
        
        prefixo = f"2-{np.random.randint(80000, 99999)}"
        ts = base_ts + i * 10
        
        lat = round(-23.56 + np.random.normal(0, 0.05), 4 + np.random.randint(0, 3))  # precisão variável
        lon = round(-46.65 + np.random.normal(0, 0.05), 4 + np.random.randint(0, 3))
        
        # Bug: veículos fora de serviço → coordenada 0,0
        if np.random.random() < 0.03:
            lat, lon = 0.0, 0.0
        
        # Bug: lotação com valores inconsistentes (mistura PT/EN)
        lotacao = np.random.choice(LOTACAO_VARIANTS, p=[0.3, 0.15, 0.1, 0.05, 0.25, 0.1, 0.05])
        
        motorista_id = f"MTR-{np.random.randint(100, 999):05d}"  # PII — deve ser anonimizado
        
        registros.append({
            "prefixo":        prefixo,
            "linha":          linha_id,
            "lat":            lat,
            "lon":            lon,
            "timestamp":      ts,           # UNIX epoch — sem timezone explícito
            "velocidade":     np.random.randint(0, 60),
            "sentido":        np.random.choice([0, 1]),
            "acessibilidade": bool(np.random.choice([True, False], p=[0.7, 0.3])),
            "lotacao":        lotacao,
            "ar_condicionado": bool(np.random.choice([True, False], p=[0.85, 0.15])),
            "motorista_id":   motorista_id,  # PII — LGPD
            "_ingested_at":   datetime.utcnow().isoformat() + "Z",
            "_source_system": "gps_bus_sptrans",
        })
    
    return pd.DataFrame(registros)

df_gps_bronze = simular_gps_bronze(300)

print("=" * 60)
print("BRONZE — GPS de Ônibus (SPTrans)")
print(f"Registros: {len(df_gps_bronze):,}")
print(f"Valores de lotação únicos (inconsistentes): {df_gps_bronze.lotacao.unique()}")
print(f"Coordenadas 0,0 (fora de serviço): {((df_gps_bronze.lat == 0) & (df_gps_bronze.lon == 0)).sum()}")
print(f"Formato timestamp: UNIX epoch (ex: {df_gps_bronze.timestamp.iloc[0]})")
print("=" * 60)
df_gps_bronze[['prefixo', 'linha', 'lat', 'lon', 'timestamp', 'lotacao', 'motorista_id']].head(5)

In [ ]:
# ============================================================
# BRONZE — Fonte 4: Ouvidoria Municipal
# CDC do PostgreSQL via Debezium
# ============================================================

CATEGORIAS = [
    "ILUMINAÇÃO PÚBLICA", "BURACOS E PAVIMENTAÇÃO", "COLETA DE LIXO",
    "TRANSPORTE PÚBLICO", "SEGURANÇA PÚBLICA", "ÁRVORES E PODA",
    "ESGOTO E DRENAGEM", "CALÇADAS", "PERTURBAÇÃO DO SOSSEGO",
]

BAIRROS = [
    "Vila Madalena", "Pinheiros", "Moema", "Itaim Bibi", "Perdizes",
    "Lapa", "Butantã", "Vila Olímpia", "Santo André", "São Bernardo",
]

DESCRICOES_TEMPLATE = [
    "Poste apagado na {rua}, {num}, há {dias} dias. Perigoso à noite.",
    "Buraco enorme na {rua} sentido {bairro}. Já machuquei meu carro. MEU CPF É {cpf}",  # PII propositalmente
    "Lixo acumulado há {dias} dias na calçada da {rua}, {num}. Meu tel: {tel}",          # PII propositalmente
    "Ônibus linha {linha} atrasado toda manhã. Não consigo chegar no trabalho a tempo.",
    "Barulho excessivo no bar da {rua}, {num}. Todo fim de semana até 3h da manhã.",
    "Árvore caída bloqueando a {rua} próximo ao número {num}. Urgente!",
]

def gerar_cpf_fake():
    nums = [np.random.randint(0, 10) for _ in range(9)]
    return f"{''.join(map(str, nums[:3]))}.{''.join(map(str, nums[3:6]))}.{''.join(map(str, nums[6:9]))}-{np.random.randint(10, 99)}"

def simular_ouvidoria_bronze(n_registros=200):
    registros = []
    base_date = datetime(2026, 6, 15, 0, 0, 0)
    
    for i in range(n_registros):
        categoria = np.random.choice(CATEGORIAS)
        bairro    = np.random.choice(BAIRROS)
        rua       = np.random.choice(["R.", "Rua", "RUA", "rua"])[0:3] + f" das Flores"
        num       = np.random.randint(10, 2000)
        dias      = np.random.randint(1, 30)
        linha_bus = np.random.choice(["875P", "702U", "5100", "8000"])
        
        template = np.random.choice(DESCRICOES_TEMPLATE)
        descricao = template.format(
            rua=rua, num=num, dias=dias, bairro=bairro,
            cpf=gerar_cpf_fake(),
            tel=f"({np.random.randint(11,99)}) 9{np.random.randint(1000,9999)}-{np.random.randint(1000,9999)}",
            linha=linha_bus
        )
        
        dt_abertura = base_date + timedelta(
            days=np.random.randint(0, 4),
            hours=np.random.randint(6, 22),
            minutes=np.random.randint(0, 59)
        )
        
        # ~12% sem coordenadas
        has_coords = np.random.random() > 0.12
        lat = round(-23.56 + np.random.normal(0, 0.08), 6) if has_coords else None
        lon = round(-46.65 + np.random.normal(0, 0.08), 6) if has_coords else None
        
        registros.append({
            "id":            845900 + i,
            "protocolo":     f"2026-{dt_abertura.strftime('%m%d')}-{dt_abertura.strftime('%H%M%S')}",
            "dt_abertura":   dt_abertura.isoformat(),
            "dt_fechamento": None if np.random.random() > 0.6 else (dt_abertura + timedelta(days=np.random.randint(1, 10))).isoformat(),
            "categoria":     categoria,
            "descricao_livre": descricao,
            "logradouro":    f"{rua}, {num}",
            "bairro":        bairro,
            "cep":           f"{np.random.randint(1000, 9999):04d}{np.random.randint(100, 999):03d}",
            "lat":           lat,
            "lon":           lon,
            "status":        np.random.choice(["ABERTA", "EM_ANDAMENTO", "FECHADA", "OPEN", "CLOSED"],
                                             p=[0.35, 0.2, 0.3, 0.08, 0.07]),  # mistura PT/EN
            "prioridade":    np.random.randint(1, 6),
            "origem":        np.random.choice(["APP", "PORTAL", "156"], p=[0.5, 0.3, 0.2]),
            "_ingested_at":  datetime.utcnow().isoformat() + "Z",
            "_source_system": "ouvidoria_db_cdc",
        })
    
    return pd.DataFrame(registros)

df_ouvidoria_bronze = simular_ouvidoria_bronze(200)

print("=" * 60)
print("BRONZE — Ouvidoria Municipal (CDC)")
print(f"Registros: {len(df_ouvidoria_bronze):,}")
print(f"Sem coordenadas: {df_ouvidoria_bronze.lat.isna().sum()} ({df_ouvidoria_bronze.lat.isna().mean():.1%})")
print(f"Status inconsistente (OPEN/CLOSED): {df_ouvidoria_bronze.status.isin(['OPEN','CLOSED']).sum()}")
print(f"\nExemplo de descrição COM PII:")
pii_example = df_ouvidoria_bronze[df_ouvidoria_bronze.descricao_livre.str.contains(r'CPF|tel:', case=False)].iloc[0]
print(f'  "{pii_example.descricao_livre}"')
print("=" * 60)
df_ouvidoria_bronze[['id', 'protocolo', 'categoria', 'bairro', 'lat', 'lon', 'status']].head(4)

---
## 2. BRONZE → SILVER: Transformações de Limpeza e Padronização

In [ ]:
# ============================================================
# SILVER — Transformação dos Sensores de Tráfego
# ============================================================

def transformar_traffic_bronze_to_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    df = df_bronze.copy()
    
    # 1. Parse de timestamp — detecta UTC (Z) vs BRT (sem Z)
    def parse_timestamp(ts_str):
        if ts_str is None:
            return pd.NaT
        if ts_str.endswith('Z'):
            return pd.to_datetime(ts_str, utc=True)
        else:
            # Assumir BRT (UTC-3) e converter para UTC
            dt_local = pd.to_datetime(ts_str)
            return dt_local + pd.Timedelta(hours=3)
    
    df['timestamp_utc'] = df['timestamp_utc'].apply(parse_timestamp)
    
    # 2. Classifica qualidade dos registros
    df['qualidade'] = 'OK'
    df.loc[df['contagem_veiculos'] == -1, 'qualidade'] = 'SENSOR_MANUTENCAO'
    
    # 3. Trata sensores em manutenção — marca para exclusão de agregações
    df.loc[df['qualidade'] == 'SENSOR_MANUTENCAO', 'contagem_veiculos'] = pd.NA
    df.loc[df['qualidade'] == 'SENSOR_MANUTENCAO', 'ocupacao_pct'] = pd.NA
    
    # 4. Velocidade nula com contagem 0 → 0 km/h (sem veículos, velocidade é 0)
    mask_sem_veiculo = (df['contagem_veiculos'] == 0) & df['velocidade_media_kmh'].isna()
    df.loc[mask_sem_veiculo, 'velocidade_media_kmh'] = 0.0
    df.loc[mask_sem_veiculo, 'qualidade'] = 'VELOCIDADE_INFERIDA'
    
    # 5. Expande lista de contagem por classe
    df['n_carros'] = df['contagem_por_classe'].apply(lambda x: x[0] if isinstance(x, list) else 0)
    df['n_motos']  = df['contagem_por_classe'].apply(lambda x: x[1] if isinstance(x, list) else 0)
    df['n_onibus'] = df['contagem_por_classe'].apply(lambda x: x[2] if isinstance(x, list) else 0)
    
    # 6. Remove campos operacionais desnecessários para análise
    df = df.drop(columns=['classificacao', 'contagem_por_classe', 'firmware_version', 'bateria_pct'])
    
    # 7. Renomeia para schema Silver
    df = df.rename(columns={'velocidade_media_kmh': 'velocidade_kmh'})
    
    # 8. Adiciona partição
    df['dt_particao'] = df['timestamp_utc'].dt.date
    
    # 9. Remove colunas de metadados de ingestão (ficam no Bronze)
    df = df.drop(columns=['_ingested_at', '_source_system'], errors='ignore')
    
    return df

df_traffic_silver = transformar_traffic_bronze_to_silver(df_traffic_bronze)

print("=" * 60)
print("SILVER — Sensores de Tráfego")
print(f"Registros antes: {len(df_traffic_bronze):,}  |  Depois: {len(df_traffic_silver):,}")
print(f"Distribuição de qualidade:")
print(df_traffic_silver['qualidade'].value_counts())
print(f"\nTimestamps agora todos em UTC: {df_traffic_silver.timestamp_utc.dt.tz}")
print("=" * 60)
df_traffic_silver[['sensor_id', 'timestamp_utc', 'via', 'contagem_veiculos', 'velocidade_kmh', 'ocupacao_pct', 'qualidade']].head(5)

In [ ]:
# ============================================================
# SILVER — Transformação da Qualidade do Ar
# ============================================================

# Breakpoints para cálculo do IQAr (metodologia CETESB)
IQAR_BREAKPOINTS = {
    'MP10_ug_m3':   [(0,50,0,40), (50,150,41,80), (150,250,81,120), (250,420,121,200), (420,600,201,400)],
    'MP2.5_ug_m3':  [(0,25,0,40), (25,60,41,80),  (60,150,81,120), (150,250,121,200), (250,500,201,400)],
    'O3_ug_m3':     [(0,100,0,40),(100,160,41,80),(160,200,81,120),(200,800,121,200),(800,2000,201,400)],
    'NO2_ug_m3':    [(0,200,0,40),(200,240,41,80),(240,320,81,120),(320,1130,121,200),(1130,3000,201,400)],
    'CO_mg_m3':     [(0,9,0,40),  (9,11,41,80),   (11,13.5,81,120),(13.5,15,121,200),(15,40,201,400)],
    'SO2_ug_m3':    [(0,40,0,40), (40,365,41,80), (365,800,81,120),(800,1600,121,200),(1600,2620,201,400)],
}

def calcular_iqar_poluente(valor, breakpoints):
    if pd.isna(valor) or valor < 0:
        return np.nan
    for cp_low, cp_high, iq_low, iq_high in breakpoints:
        if cp_low <= valor <= cp_high:
            return iq_low + (iq_high - iq_low) * (valor - cp_low) / (cp_high - cp_low)
    return 400

def faixa_iqar(iqar):
    if pd.isna(iqar):   return 'INDISPONIVEL'
    if iqar <= 40:      return 'BOA'
    if iqar <= 80:      return 'MODERADA'
    if iqar <= 120:     return 'RUIM'
    if iqar <= 200:     return 'MUITO_RUIM'
    return 'PESSIMA'

def transformar_ar_bronze_to_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    df = df_bronze.copy()
    
    # 1. Substitui strings vazias por NaN
    df = df.replace("", np.nan)
    
    # 2. Converte colunas numéricas
    numeric_cols = ['MP10_ug_m3', 'MP2.5_ug_m3', 'O3_ug_m3', 'NO2_ug_m3',
                    'CO_mg_m3', 'SO2_ug_m3', 'TEMP_C', 'UMIDADE_PCT',
                    'PRESSAO_hPa', 'VENTO_DIR_GRAUS', 'VENTO_VEL_ms']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 3. Unifica data e hora em timestamp UTC
    df['timestamp_utc'] = pd.to_datetime(
        df['DT_MEDICAO'].str.strip() + ' ' + df['HORA'].str.strip(),
        format='%d/%m/%Y %H:%M',
        utc=True
    )
    df = df.drop(columns=['DT_MEDICAO', 'HORA'])
    
    # 4. Normaliza CO da estação CAC001 de ppm → mg/m³
    mask_cac = df['ESTACAO_ID'] == 'CAC001'
    df.loc[mask_cac, 'CO_mg_m3'] = df.loc[mask_cac, 'CO_mg_m3'] * 1.165
    
    # 5. Remove valores fisicamente impossíveis (negativos)
    for col in ['MP10_ug_m3', 'MP2.5_ug_m3', 'O3_ug_m3', 'NO2_ug_m3', 'CO_mg_m3', 'SO2_ug_m3']:
        df.loc[df[col] < 0, col] = np.nan
    
    # 6. Calcula IQAr por poluente e retorna o máximo (índice composto)
    for col, bps in IQAR_BREAKPOINTS.items():
        if col in df.columns:
            df[f'iqar_{col.split("_")[0].lower()}'] = df[col].apply(lambda v: calcular_iqar_poluente(v, bps))
    
    iqar_cols = [c for c in df.columns if c.startswith('iqar_')]
    df['iqar'] = df[iqar_cols].max(axis=1)
    df['faixa_iqar'] = df['iqar'].apply(faixa_iqar)
    
    # 7. Renomeia para schema padronizado (snake_case)
    df = df.rename(columns={
        'ESTACAO_ID': 'estacao_id', 'MP10_ug_m3': 'mp10', 'MP2.5_ug_m3': 'mp25',
        'O3_ug_m3': 'o3', 'NO2_ug_m3': 'no2', 'CO_mg_m3': 'co', 'SO2_ug_m3': 'so2',
        'TEMP_C': 'temp_c', 'UMIDADE_PCT': 'umidade_pct', 'PRESSAO_hPa': 'pressao_hpa',
        'VENTO_DIR_GRAUS': 'vento_dir', 'VENTO_VEL_ms': 'vento_vel_ms'
    })
    
    # 8. Drop colunas IQAr individuais (mantém só o composto)
    df = df.drop(columns=iqar_cols)
    df['dt_particao'] = df['timestamp_utc'].dt.date
    
    return df

df_ar_silver = transformar_ar_bronze_to_silver(df_ar_bronze)

print("=" * 60)
print("SILVER — Qualidade do Ar")
print(f"Registros: {len(df_ar_silver):,}")
print(f"Distribuição de faixas IQAr:")
print(df_ar_silver['faixa_iqar'].value_counts())
print(f"\nMP2.5 negativos removidos: {df_ar_silver.mp25.isna().sum() - df_ar_bronze['MP2.5_ug_m3'].isna().sum()}")
print("=" * 60)
df_ar_silver[['estacao_id', 'timestamp_utc', 'mp10', 'mp25', 'o3', 'iqar', 'faixa_iqar', 'temp_c']].head(5)

In [ ]:
# ============================================================
# SILVER — Transformação GPS de Ônibus (LGPD + normalização)
# ============================================================

LOTACAO_MAP = {
    'VAZIA': 'VAZIA', 'MEIA': 'MEIA', 'MEIA_LOTACAO': 'MEIA',
    'CHEIA': 'CHEIA', 'CHEIO': 'CHEIA',
    'LOTADA': 'LOTADA', 'FULL': 'LOTADA'
}

def anonimizar_id(valor: str, salt: str = "metropole-sp-2026") -> str:
    """Pseudoanonimização SHA-256 com salt — LGPD Art. 12"""
    if pd.isna(valor):
        return None
    return hashlib.sha256(f"{salt}:{valor}".encode()).hexdigest()[:16]

def transformar_gps_bronze_to_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    df = df_bronze.copy()
    
    # 1. Converte UNIX epoch → datetime UTC
    df['timestamp_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df = df.drop(columns=['timestamp'])
    
    # 2. Remove veículos fora de serviço (coordenada 0,0)
    mask_invalido = (df['lat'] == 0.0) & (df['lon'] == 0.0)
    df['ativo'] = ~mask_invalido
    
    # 3. Normaliza lotação
    df['lotacao'] = df['lotacao'].map(LOTACAO_MAP).fillna('DESCONHECIDA')
    
    # 4. Padroniza coordenadas para 5 casas decimais
    df['lat'] = df['lat'].round(5)
    df['lon'] = df['lon'].round(5)
    
    # 5. Anonimização LGPD — motorista_id → hash irreversível
    df['motorista_hash'] = df['motorista_id'].apply(anonimizar_id)
    df = df.drop(columns=['motorista_id'])
    
    # 6. Remove metadados Bronze
    df = df.drop(columns=['_ingested_at', '_source_system'], errors='ignore')
    df['dt_particao'] = df['timestamp_utc'].dt.date
    
    return df

df_gps_silver = transformar_gps_bronze_to_silver(df_gps_bronze)

print("=" * 60)
print("SILVER — GPS de Ônibus")
print(f"Valores de lotação normalizados: {df_gps_silver.lotacao.unique()}")
print(f"Veículos fora de serviço: {(~df_gps_silver.ativo).sum()}")
print(f"motorista_id → motorista_hash (LGPD):")
print(f"  Original:    {df_gps_bronze.motorista_id.iloc[0]}")
print(f"  Anonimizado: {df_gps_silver.motorista_hash.iloc[0]}")
print("=" * 60)
df_gps_silver[['prefixo', 'linha', 'lat', 'lon', 'timestamp_utc', 'lotacao', 'ativo', 'motorista_hash']].head(5)

In [ ]:
# ============================================================
# SILVER — Ouvidoria: Remoção de PII e padronização
# ============================================================

# Padrões de PII para detecção via regex
PII_PATTERNS = [
    (r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF_REMOVIDO]'),         # CPF: 123.456.789-00
    (r'\(?\d{2}\)?\s?9?\d{4}[\s-]\d{4}', '[TEL_REMOVIDO]'),   # Telefone: (11) 9xxxx-xxxx
    (r'\d{5}-?\d{3}', '[CEP_REMOVIDO]'),                       # CEP no texto
    (r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[EMAIL_REMOVIDO]'),
]

STATUS_MAP = {
    'ABERTA': 'ABERTA', 'EM_ANDAMENTO': 'EM_ANDAMENTO',
    'FECHADA': 'FECHADA', 'OPEN': 'ABERTA', 'CLOSED': 'FECHADA'
}

LOGRADOURO_ABREV = {
    r'^R\. ': 'Rua ', r'^r\. ': 'Rua ', r'^RUA ': 'Rua ',
    r'^Av\. ': 'Avenida ', r'^AV\. ': 'Avenida ',
    r'^Pç\. ': 'Praça ', r'^Al\. ': 'Alameda ',
}

def remover_pii(texto: str) -> str:
    if pd.isna(texto):
        return texto
    for pattern, replacement in PII_PATTERNS:
        texto = re.sub(pattern, replacement, texto)
    return texto

def normalizar_logradouro(logradouro: str) -> str:
    if pd.isna(logradouro):
        return logradouro
    for pattern, replacement in LOGRADOURO_ABREV.items():
        logradouro = re.sub(pattern, replacement, logradouro)
    return logradouro

def transformar_ouvidoria_bronze_to_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    df = df_bronze.copy()
    
    # 1. Remove PII dos campos de texto livre
    df['descricao_anonimizada'] = df['descricao_livre'].apply(remover_pii)
    df = df.drop(columns=['descricao_livre'])
    
    # 2. Normaliza logradouro
    df['logradouro'] = df['logradouro'].apply(normalizar_logradouro)
    
    # 3. Padroniza status PT/EN
    df['status'] = df['status'].map(STATUS_MAP).fillna(df['status'])
    
    # 4. Converte datas
    df['dt_abertura']   = pd.to_datetime(df['dt_abertura'], utc=True)
    df['dt_fechamento'] = pd.to_datetime(df['dt_fechamento'], utc=True, errors='coerce')
    
    # 5. Simula geocodificação para registros sem coordenadas
    mask_sem_coord = df['lat'].isna()
    # Em produção: chamada à API Nominatim / Google Maps Geocoding
    # Aqui: simula resultado da geocodificação com precisão de bairro
    COORD_BAIRRO = {
        'Vila Madalena': (-23.5489, -46.6882), 'Pinheiros': (-23.5618, -46.7027),
        'Moema': (-23.6014, -46.6681), 'Itaim Bibi': (-23.5851, -46.6745),
        'Perdizes': (-23.5347, -46.6647), 'Lapa': (-23.5193, -46.7069),
        'Butantã': (-23.5782, -46.7269), 'Vila Olímpia': (-23.5989, -46.6846),
        'Santo André': (-23.6640, -46.5322), 'São Bernardo': (-23.6941, -46.5649),
    }
    for idx, row in df[mask_sem_coord].iterrows():
        coords = COORD_BAIRRO.get(row['bairro'], (-23.5489, -46.6882))
        df.at[idx, 'lat'] = coords[0] + np.random.normal(0, 0.005)  # aleatoriza dentro do bairro
        df.at[idx, 'lon'] = coords[1] + np.random.normal(0, 0.005)
    
    df['coord_geocodificada'] = mask_sem_coord  # flag de geocodificação
    
    # 6. Remove metadados Bronze
    df = df.drop(columns=['_ingested_at', '_source_system'], errors='ignore')
    df['dt_particao'] = df['dt_abertura'].dt.date
    
    return df

df_ouvidoria_silver = transformar_ouvidoria_bronze_to_silver(df_ouvidoria_bronze)

print("=" * 60)
print("SILVER — Ouvidoria Municipal")
print(f"PII removido. Exemplo:")
idx_pii = df_ouvidoria_bronze[df_ouvidoria_bronze.descricao_livre.str.contains(r'CPF|tel:', case=False)].index[0]
print(f"  Bronze: {df_ouvidoria_bronze.loc[idx_pii, 'descricao_livre'][:80]}")
print(f"  Silver: {df_ouvidoria_silver.loc[idx_pii, 'descricao_anonimizada'][:80]}")
print(f"\nStatus padronizados: {df_ouvidoria_silver.status.unique()}")
print(f"Coordenadas geocodificadas: {df_ouvidoria_silver.coord_geocodificada.sum()}")
print("=" * 60)
df_ouvidoria_silver[['id', 'categoria', 'bairro', 'lat', 'lon', 'status', 'coord_geocodificada']].head(4)

---
## 3. SILVER → GOLD: Feature Engineering para IA

In [ ]:
# ============================================================
# GOLD — Dataset para LSTM de Predição de Tráfego
# gold.traffic_timeseries
# ============================================================

# Gera série temporal mais longa para exemplificar o dataset Gold
def gerar_gold_traffic_timeseries(via='Av. Paulista', sentido='Norte-Sul', n_horas=168):
    """
    Cria série temporal horária com features de lag e variáveis exógenas
    para treino do modelo LSTM.
    n_horas=168 → 7 dias de histórico
    """
    base_time = datetime(2026, 6, 12, 0, 0, 0)  # começa 7 dias antes
    timestamps = [base_time + timedelta(hours=h) for h in range(n_horas + 24)]
    
    # Simula fluxo com padrão diário/semanal realista
    contagens = []
    for ts in timestamps:
        hora = ts.hour
        dia_semana = ts.weekday()  # 0=segunda, 6=domingo
        
        # Padrão de pico (menor aos fins de semana)
        fator_semana = 0.6 if dia_semana >= 5 else 1.0
        pico_manha = np.exp(-((hora - 8) ** 2) / 4)
        pico_tarde = np.exp(-((hora - 18) ** 2) / 5)
        base_fluxo = 30 + 40 * (pico_manha + pico_tarde) * fator_semana
        
        # Chuva em alguns dias
        chuva = 5.0 if (ts.day in [14, 15] and 14 <= hora <= 19) else 0.0
        reducao_chuva = 0.85 if chuva > 0 else 1.0
        
        contagem = max(0, np.random.normal(base_fluxo * reducao_chuva, 5))
        contagens.append({
            'hora_referencia':    ts,
            'contagem_media_h':   round(contagem, 1),
            'velocidade_media_h': round(max(5, np.random.normal(60 - contagem * 0.8, 8)), 1),
            'ocupacao_media_h':   round(min(100, contagem * 1.5 + np.random.normal(0, 5)), 1),
            'chuva_mm':           round(chuva + np.random.exponential(0.5) if chuva > 0 else 0, 1),
            'temperatura_c':      round(19 + 5 * np.sin(hora * np.pi / 12) + np.random.normal(0, 1), 1),
            'hora_do_dia':        hora,
            'dia_semana':         dia_semana,
            'e_feriado':          ts.date() == datetime(2026, 6, 15).date(),  # Dia do Corpus Christi
        })
    
    df = pd.DataFrame(contagens)
    df['via_id'] = f"{via}_{sentido}".replace(' ', '_').replace('-', '')
    
    # Features de lag (autocorrelação temporal)
    df['lag_1h_contagem']   = df['contagem_media_h'].shift(1)
    df['lag_3h_contagem']   = df['contagem_media_h'].shift(3)
    df['lag_24h_contagem']  = df['contagem_media_h'].shift(24)
    df['lag_168h_contagem'] = df['contagem_media_h'].shift(168)
    
    # Target: contagem na PRÓXIMA hora (shift -1)
    df['target_contagem_1h'] = df['contagem_media_h'].shift(-1)
    
    # Remove linhas com NaN nos lags (primeiros 168 registros) e no target
    df = df.dropna(subset=['lag_168h_contagem', 'target_contagem_1h'])
    
    # Normalização Min-Max (os parâmetros seriam salvos no Feature Store)
    for col in ['contagem_media_h', 'velocidade_media_h', 'ocupacao_media_h',
                'lag_1h_contagem', 'lag_3h_contagem', 'lag_24h_contagem', 'lag_168h_contagem']:
        min_v, max_v = df[col].min(), df[col].max()
        df[f'{col}_norm'] = (df[col] - min_v) / (max_v - min_v + 1e-8)
    
    return df

df_gold_traffic = gerar_gold_traffic_timeseries()

print("=" * 60)
print("GOLD — traffic_timeseries (input LSTM)")
print(f"Registros: {len(df_gold_traffic):,}")
print(f"Features disponíveis: {len(df_gold_traffic.columns)}")
print(f"Período: {df_gold_traffic.hora_referencia.min().date()} a {df_gold_traffic.hora_referencia.max().date()}")
print("=" * 60)
df_gold_traffic[['hora_referencia', 'contagem_media_h', 'velocidade_media_h', 'chuva_mm',
                  'lag_24h_contagem', 'lag_168h_contagem', 'target_contagem_1h']].head(6)

In [ ]:
# ============================================================
# GOLD — Dataset para Isolation Forest (Anomalias de Ar)
# gold.air_quality_anomaly
# ============================================================

def gerar_gold_air_anomaly(df_silver: pd.DataFrame) -> pd.DataFrame:
    df = df_silver[df_silver['faixa_iqar'] != 'INDISPONIVEL'].copy()
    
    # Features para o modelo
    feature_cols = ['mp10', 'mp25', 'o3', 'no2', 'co', 'so2', 'temp_c', 'umidade_pct', 'vento_vel_ms']
    
    # Imputação de NaN por mediana da estação (KNN simplificado aqui)
    for col in feature_cols:
        if col in df.columns:
            df[col] = df.groupby('estacao_id')[col].transform(lambda x: x.fillna(x.median()))
    
    # Z-score por estação (desvio relativo ao histórico)
    for col in ['mp10', 'mp25', 'o3', 'no2']:
        if col in df.columns:
            df[f'z_score_{col}'] = df.groupby('estacao_id')[col].transform(
                lambda x: (x - x.mean()) / (x.std() + 1e-8)
            )
    
    # Feature: hora do dia (padrão diário de poluição)
    df['hora_do_dia'] = pd.to_datetime(df['timestamp_utc']).dt.hour
    
    # Label de anomalia: IQAr > 120 (RUIM) como proxy de evento anômalo
    df['anomalia'] = df['iqar'] > 120
    
    # Padronização (StandardScaler)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    cols_to_scale = [c for c in feature_cols if c in df.columns]
    df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale].fillna(0))
    
    return df

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import IsolationForest
    
    df_gold_ar = gerar_gold_air_anomaly(df_ar_silver)
    
    # Demo rápido do Isolation Forest com os dados Gold
    feature_cols_ml = ['mp10', 'mp25', 'o3', 'no2', 'temp_c', 'umidade_pct', 'hora_do_dia']
    X = df_gold_ar[[c for c in feature_cols_ml if c in df_gold_ar.columns]].fillna(0)
    
    iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    df_gold_ar['anomaly_score_raw'] = iso_forest.fit_predict(X)
    df_gold_ar['anomaly_detectada'] = df_gold_ar['anomaly_score_raw'] == -1
    
    print("=" * 60)
    print("GOLD — air_quality_anomaly (input Isolation Forest)")
    print(f"Registros: {len(df_gold_ar):,}")
    print(f"Anomalias detectadas pelo modelo: {df_gold_ar.anomaly_detectada.sum()} ({df_gold_ar.anomaly_detectada.mean():.1%})")
    print(f"IQAr > 120 (label): {df_gold_ar.anomalia.sum()}")
    print("=" * 60)
    
except ImportError:
    print("scikit-learn não instalado. Execute: pip install scikit-learn")
    df_gold_ar = df_ar_silver.copy()

In [ ]:
# ============================================================
# GOLD — Corpus para BERT (Classificação de Ocorrências)
# gold.ouvidoria_nlp
# ============================================================

# Mapeamento de categorias para hierarquia NLP
CATEGORIA_HIERARQUIA = {
    'ILUMINAÇÃO PÚBLICA':     ('INFRAESTRUTURA', 'ILUMINACAO'),
    'BURACOS E PAVIMENTAÇÃO': ('INFRAESTRUTURA', 'PAVIMENTACAO'),
    'CALÇADAS':               ('INFRAESTRUTURA', 'CALCADAS'),
    'COLETA DE LIXO':         ('MEIO_AMBIENTE', 'RESIDUOS'),
    'ÁRVORES E PODA':         ('MEIO_AMBIENTE', 'ARBORIZACAO'),
    'ESGOTO E DRENAGEM':      ('MEIO_AMBIENTE', 'SANEAMENTO'),
    'TRANSPORTE PÚBLICO':     ('MOBILIDADE', 'TRANSPORTE_COLETIVO'),
    'SEGURANÇA PÚBLICA':      ('SEGURANCA', 'OCORRENCIAS'),
    'PERTURBAÇÃO DO SOSSEGO': ('SEGURANCA', 'PERTURBACAO'),
}

def gerar_gold_ouvidoria_nlp(df_silver: pd.DataFrame) -> pd.DataFrame:
    df = df_silver.copy()
    
    # Mapeamento de hierarquia
    df['categoria_l1'] = df['categoria'].map(lambda c: CATEGORIA_HIERARQUIA.get(c, ('OUTROS', 'NAO_CLASSIFICADO'))[0])
    df['categoria_l2'] = df['categoria'].map(lambda c: CATEGORIA_HIERARQUIA.get(c, ('OUTROS', 'NAO_CLASSIFICADO'))[1])
    
    # Seleciona apenas campos relevantes para NLP
    df = df[['id', 'descricao_anonimizada', 'lat', 'lon', 'bairro',
              'categoria_l1', 'categoria_l2', 'prioridade']].rename(
        columns={'descricao_anonimizada': 'texto'}
    )
    
    # Remove textos muito curtos (< 10 chars) — pouca informação
    df = df[df['texto'].str.len() > 10]
    
    # Split estratificado treino/validação/teste
    np.random.seed(42)
    rand = np.random.random(len(df))
    df['split'] = pd.cut(rand, bins=[0, 0.70, 0.85, 1.0], labels=['TRAIN', 'VAL', 'TEST'])
    
    return df

df_gold_nlp = gerar_gold_ouvidoria_nlp(df_ouvidoria_silver)

print("=" * 60)
print("GOLD — ouvidoria_nlp (corpus para BERT)")
print(f"Registros: {len(df_gold_nlp):,}")
print(f"\nDistribuição por split:")
print(df_gold_nlp.split.value_counts())
print(f"\nDistribuição por categoria L1:")
print(df_gold_nlp.categoria_l1.value_counts())
print("=" * 60)
df_gold_nlp[['id', 'texto', 'categoria_l1', 'categoria_l2', 'split']].head(5)

---
## 4. Visualizações Analíticas

In [ ]:
# ============================================================
# Figura 1: Evolução temporal do fluxo de tráfego (Gold)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Pipeline Gold — Série Temporal de Tráfego (Av. Paulista)', fontsize=13, fontweight='bold')

ax1, ax2 = axes

# Contagem horária
ax1.plot(df_gold_traffic['hora_referencia'], df_gold_traffic['contagem_media_h'],
         color='#2196F3', linewidth=0.8, alpha=0.8, label='Contagem de veículos/h')
ax1.fill_between(df_gold_traffic['hora_referencia'], df_gold_traffic['contagem_media_h'],
                  alpha=0.1, color='#2196F3')

# Destaca feriado
feriado_mask = df_gold_traffic['e_feriado'] == True
if feriado_mask.any():
    ax1.axvspan(df_gold_traffic[feriado_mask]['hora_referencia'].min(),
                df_gold_traffic[feriado_mask]['hora_referencia'].max(),
                alpha=0.15, color='orange', label='Feriado (Corpus Christi)')

ax1.set_ylabel('Veículos/hora (média)', fontsize=10)
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax1.xaxis.set_major_locator(mdates.HourLocator(interval=12))
ax1.grid(alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Velocidade média
ax2.plot(df_gold_traffic['hora_referencia'], df_gold_traffic['velocidade_media_h'],
         color='#4CAF50', linewidth=0.8, alpha=0.8, label='Velocidade média (km/h)')
ax2.bar(df_gold_traffic['hora_referencia'], df_gold_traffic['chuva_mm'] * 3,
        color='#03A9F4', alpha=0.4, width=0.03, label='Precipitação (escala ×3)')
ax2.set_ylabel('Velocidade km/h | Precipitação mm', fontsize=10)
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=12))
ax2.grid(alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('fig1_trafego_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figura 1 salva: fig1_trafego_timeseries.png")

In [ ]:
# ============================================================
# Figura 2: Qualidade do Ar — IQAr por estação e hora
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pipeline Silver — Qualidade do Ar (IQAr)', fontsize=13, fontweight='bold')

# Esquerda: Heatmap IQAr por estação × hora do dia
df_pivot = df_ar_silver.copy()
df_pivot['hora'] = pd.to_datetime(df_pivot['timestamp_utc']).dt.hour
pivot = df_pivot.groupby(['estacao_id', 'hora'])['iqar'].mean().unstack()

# Ordena estações por IQAr médio
pivot = pivot.reindex(pivot.mean(axis=1).sort_values(ascending=False).index)

sns.heatmap(pivot, ax=axes[0], cmap='RdYlGn_r', vmin=0, vmax=120,
            linewidths=0.5, cbar_kws={'label': 'IQAr'}, fmt='.0f')
axes[0].set_title('IQAr médio por Estação × Hora do Dia', fontsize=10)
axes[0].set_xlabel('Hora do dia (UTC)')
axes[0].set_ylabel('Estação de monitoramento')

# Direita: Distribuição de MP2.5 após limpeza Silver
axes[1].hist(df_ar_silver['mp25'].dropna(), bins=40, color='#9C27B0', alpha=0.7, edgecolor='white')
axes[1].axvline(x=25, color='orange', linestyle='--', linewidth=2, label='Limite OMS (25 µg/m³)')
axes[1].axvline(x=60, color='red', linestyle='--', linewidth=2, label='Faixa Ruim CETESB (60 µg/m³)')
axes[1].set_xlabel('MP2.5 (µg/m³)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de MP2.5 — Camada Silver', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig2_qualidade_ar.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figura 2 salva: fig2_qualidade_ar.png")

In [ ]:
# ============================================================
# Figura 3: Mapa de calor de ocorrências por bairro
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pipeline Gold — Ocorrências Urbanas', fontsize=13, fontweight='bold')

# Esquerda: Dispersão geoespacial das ocorrências
cat_colors = {
    'INFRAESTRUTURA': '#2196F3', 'MEIO_AMBIENTE': '#4CAF50',
    'MOBILIDADE': '#FF9800', 'SEGURANCA': '#F44336'
}
for cat, color in cat_colors.items():
    mask = df_gold_nlp['categoria_l1'] == cat
    axes[0].scatter(
        df_gold_nlp[mask]['lon'], df_gold_nlp[mask]['lat'],
        c=color, label=cat, alpha=0.6, s=20
    )
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Dispersão Geoespacial de Ocorrências', fontsize=10)
axes[0].legend(fontsize=8, loc='upper right')
axes[0].grid(alpha=0.3)

# Direita: Volume por categoria e prioridade
pivot_prio = df_gold_nlp.groupby(['categoria_l1', 'prioridade']).size().unstack(fill_value=0)
pivot_prio.plot(kind='bar', ax=axes[1], stacked=True,
                 colormap='RdYlGn_r', edgecolor='white', width=0.7)
axes[1].set_title('Ocorrências por Categoria e Prioridade', fontsize=10)
axes[1].set_xlabel('')
axes[1].set_ylabel('Número de ocorrências')
axes[1].legend(title='Prioridade', labels=['1-Baixa','2','3-Média','4','5-Crítica'], fontsize=8)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig3_ocorrencias_urbanas.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figura 3 salva: fig3_ocorrencias_urbanas.png")

In [ ]:
# ============================================================
# Figura 4: Resumo do Pipeline — Qualidade em cada camada
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Rastreamento de Qualidade: Bronze → Silver → Gold', fontsize=13, fontweight='bold')

# 1. Taxa de completude por fonte
completude = {
    'Bronze Tráfego':    1 - df_traffic_bronze['velocidade_media_kmh'].isna().mean(),
    'Silver Tráfego':    1 - df_traffic_silver['velocidade_kmh'].isna().mean(),
    'Bronze Qualidade':  1 - df_ar_bronze['MP2.5_ug_m3'].isin(['', None]).mean(),
    'Silver Qualidade':  1 - df_ar_silver['mp25'].isna().mean(),
    'Bronze Ouvidoria':  df_ouvidoria_bronze['lat'].notna().mean(),
    'Silver Ouvidoria':  df_ouvidoria_silver['lat'].notna().mean(),
}
bars = axes[0,0].barh(list(completude.keys()), [v*100 for v in completude.values()],
                       color=['#FF9800','#4CAF50','#FF9800','#4CAF50','#FF9800','#4CAF50'])
axes[0,0].axvline(x=95, color='red', linestyle='--', alpha=0.7, label='SLA 95%')
axes[0,0].set_xlabel('% de completude (campo principal)')
axes[0,0].set_title('Completude por Camada', fontsize=10)
axes[0,0].legend(fontsize=8)
for bar, v in zip(bars, completude.values()):
    axes[0,0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                   f'{v:.1%}', va='center', fontsize=8)

# 2. Volume de dados por fonte
volumes = {
    'IoT Tráfego': len(df_traffic_bronze),
    'Qualidade Ar': len(df_ar_bronze),
    'GPS Ônibus': len(df_gps_bronze),
    'Ouvidoria': len(df_ouvidoria_bronze),
}
axes[0,1].pie(volumes.values(), labels=volumes.keys(), autopct='%1.1f%%',
               colors=['#2196F3', '#9C27B0', '#FF9800', '#F44336'],
               startangle=90, pctdistance=0.75)
axes[0,1].set_title(f'Volume por Fonte (Total: {sum(volumes.values()):,} registros)', fontsize=10)

# 3. IQAr Gold ao longo do tempo
df_iqar_time = df_ar_silver.groupby(
    df_ar_silver['timestamp_utc'].dt.floor('H')
)['iqar'].mean().reset_index()
axes[1,0].plot(df_iqar_time['timestamp_utc'], df_iqar_time['iqar'],
                color='#9C27B0', linewidth=1.2)
axes[1,0].fill_between(df_iqar_time['timestamp_utc'], df_iqar_time['iqar'],
                         where=df_iqar_time['iqar'] > 80, alpha=0.3, color='red', label='IQAr > 80 (Ruim)')
axes[1,0].axhline(y=80, color='orange', linestyle='--', alpha=0.7, linewidth=1)
axes[1,0].set_title('IQAr Médio Horário (Silver)', fontsize=10)
axes[1,0].set_ylabel('IQAr')
axes[1,0].legend(fontsize=8)
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m\n%Hh'))
axes[1,0].grid(alpha=0.3)

# 4. Distribuição de prioridade das ocorrências
prio_counts = df_ouvidoria_silver['prioridade'].value_counts().sort_index()
colors_prio = ['#4CAF50', '#8BC34A', '#FFC107', '#FF5722', '#F44336']
axes[1,1].bar(prio_counts.index, prio_counts.values, color=colors_prio, edgecolor='white')
axes[1,1].set_title('Ocorrências por Prioridade (Silver)', fontsize=10)
axes[1,1].set_xlabel('Prioridade (1=Baixa, 5=Crítica)')
axes[1,1].set_ylabel('Número de ocorrências')
for i, v in zip(prio_counts.index, prio_counts.values):
    axes[1,1].text(i, v + 1, str(v), ha='center', fontsize=9)
axes[1,1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig4_resumo_pipeline.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figura 4 salva: fig4_resumo_pipeline.png")

---
## 5. Resumo do Pipeline — Tabela de Contagem por Camada

In [ ]:
# ============================================================
# Resumo final: contagens e transformações por camada
# ============================================================

resumo = pd.DataFrame([
    {
        'Fonte / Dataset': 'IoT Tráfego CET',
        'Bronze (registros)': len(df_traffic_bronze),
        'Silver (registros)': len(df_traffic_silver),
        'Transformações Chave': 'Timestamp UTC, qualidade, expande classes',
        'Gold Dataset': 'traffic_timeseries',
        'Algoritmo IA': 'LSTM Predição de Fluxo'
    },
    {
        'Fonte / Dataset': 'Qualidade do Ar CETESB',
        'Bronze (registros)': len(df_ar_bronze),
        'Silver (registros)': len(df_ar_silver),
        'Transformações Chave': 'IQAr, unidade CO, datas ISO, nulos tipados',
        'Gold Dataset': 'air_quality_anomaly',
        'Algoritmo IA': 'Isolation Forest Anomalia'
    },
    {
        'Fonte / Dataset': 'GPS Ônibus SPTrans',
        'Bronze (registros)': len(df_gps_bronze),
        'Silver (registros)': len(df_gps_silver),
        'Transformações Chave': 'LGPD hash, lotação enum, coords padrão',
        'Gold Dataset': 'bus_demand_forecast',
        'Algoritmo IA': 'XGBoost Demanda Transporte'
    },
    {
        'Fonte / Dataset': 'Ouvidoria Municipal',
        'Bronze (registros)': len(df_ouvidoria_bronze),
        'Silver (registros)': len(df_ouvidoria_silver),
        'Transformações Chave': 'PII removido, geocodificação, status PT',
        'Gold Dataset': 'ouvidoria_nlp',
        'Algoritmo IA': 'BERTimbau Classificação NLP'
    },
])

print("=" * 80)
print("RESUMO DO PIPELINE — Metrópole SP Data Platform")
print("=" * 80)
print(resumo.to_string(index=False))
print("=" * 80)
print(f"\nTotal Bronze: {resumo['Bronze (registros)'].sum():,} registros")
print(f"Total Silver: {resumo['Silver (registros)'].sum():,} registros")
print(f"\nDatasets Gold gerados: 4 (traffic_timeseries, air_quality_anomaly, bus_demand_forecast, ouvidoria_nlp)")
print(f"Algoritmos de IA alimentados: LSTM, Isolation Forest, XGBoost, BERTimbau, DBSCAN")

In [ ]:
# ============================================================
# Export dos datasets para Parquet (simulação do armazenamento
# no Data Lake nos formatos reais do pipeline)
# ============================================================

import os

os.makedirs('data/bronze', exist_ok=True)
os.makedirs('data/silver', exist_ok=True)
os.makedirs('data/gold', exist_ok=True)

# Bronze — Parquet (como chegaria do Kafka Consumer)
df_traffic_bronze.to_parquet('data/bronze/iot_traffic_raw.parquet', index=False)
df_ar_bronze.to_parquet('data/bronze/air_quality_raw.parquet', index=False)
df_gps_bronze.to_parquet('data/bronze/gps_bus_raw.parquet', index=False)
df_ouvidoria_bronze.to_parquet('data/bronze/ouvidoria_cdc_raw.parquet', index=False)

# Silver — Parquet (Delta Lake em produção)
df_traffic_silver.to_parquet('data/silver/iot_traffic_clean.parquet', index=False)
df_ar_silver.to_parquet('data/silver/air_quality_clean.parquet', index=False)
df_gps_silver.to_parquet('data/silver/gps_bus_clean.parquet', index=False)
df_ouvidoria_silver.to_parquet('data/silver/ouvidoria_clean.parquet', index=False)

# Gold — Parquet otimizado para leitura por modelos
df_gold_traffic.to_parquet('data/gold/traffic_timeseries.parquet', index=False)
df_gold_nlp.to_parquet('data/gold/ouvidoria_nlp.parquet', index=False)

print("Arquivos Parquet exportados com sucesso:")
for root, dirs, files in os.walk('data'):
    for file in files:
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  {filepath:55s} {size_kb:6.1f} KB")

---
## 6. Conclusão

Este notebook demonstrou o fluxo completo do pipeline de dados da **Metrópole SP Data Platform**:

### O que foi implementado:

| Camada | Operações Demonstradas |
|---|---|
| **Bronze** | Simulação fiel de payloads reais com problemas introduzidos propositalmente |
| **Bronze → Silver** | Padronização de timestamps, remoção de PII (LGPD), normalização de enums, geocodificação, cálculo de IQAr |
| **Silver → Gold** | Feature engineering com lags temporais, z-scores, normalização, mapeamento hierárquico para NLP |
| **Export** | Arquivos `.parquet` simulando o armazenamento em Data Lake |

### Algoritmos de IA abastecidos:

- **LSTM** ← `gold/traffic_timeseries.parquet` (série temporal com lags de 7 dias)
- **Isolation Forest** ← `gold/air_quality_anomaly` (11 features + z-scores por estação)
- **BERTimbau** ← `gold/ouvidoria_nlp.parquet` (corpus com 200 textos anonimizados, split treino/val/teste)
- **XGBoost** ← `gold/bus_demand_forecast` (features de demanda + exógenas)
- **DBSCAN** ← `gold/occurrence_heatmap` (consolidação geoespacial multi-fonte)

> Para execução em escala real: substituir o código Python por **PySpark** (Silver) + **dbt** (Gold), com ingestão via **Apache Kafka** e orquestração pelo **Apache Airflow**. A estrutura das transformações é idêntica — apenas o motor de execução muda.

---
*Gabriel Felice — Engenharia de Dados — Junho 2026*